# ICP-count regression - does the leading-var rise scale with connections?

**Decisive check #1 of the proxy decomposition** (harmonics workstream: `clients/ea/harmonics/consultation-2026/analysis/PROXY_DECOMPOSITION_20260806.md`). Dave's proposal, 6 Aug 2026; first run same evening; this notebook is the canonical version (supersedes `icp_regression_20260806.py`, kept for provenance). **Relocated to the IEEE paper directory 6 Aug pm (Dave's call)** - this regression is core evidence for the paper's organic-mechanism correction (the demand fleet is *adding* distributed capacitance, not merely drawing fewer lagging vars). Inputs are read in place from the harmonics resonance-screen panel and the power-factor contamination flags.

**Question.** The 2013-25 rise in overnight net leading vars at clean GXPs: does it scale with the number of connections (ICPs) behind each bus? The device-fleet story (EMI filter X-capacitors etc.) predicts a slope of roughly 100-250 VAr per connection with zero intercept. A GXP-level metering artefact has no mechanism to scale with connections - its prediction is the mean-only model.

**Pre-registered decision rule** (fixed before first computation, 6 Aug pm): SUPPORTED if the 95% CI on the slope excludes 0, the slope lies in [50, 400] VAr/ICP, and R^2 > 0.15. ARTEFACT-FAVOURED if the CI includes 0. Otherwise inconclusive.

**Data.** ICP counts: EMI `Retail/Datasets/MarketStructure/20260630_MarketShareTrendsByRootNSP.csv` (downloaded 6 Aug 2026; monthly ICP count by root NSP x retailer, Dec 2003 - Jun 2026; archived gzipped in `data_raw/`). Root NSP = POC + network + suffix; ICPs are summed per POC across networks and retailers, so embedded networks roll up to the grid POC. Screen panel: `screen_by_gxp_year.csv` (night-median net leading MVAr). Clean cohort: power-factor `contamination_flag.csv`.

Assertion battery throughout = drift guard; run end-to-end after any data refresh.

> **Version note (8 Aug 2026).** This notebook backs Section V of the paper
> *"From Lagging to Leading: The Measured Emergence of Standing Capacitance
> Behind Consumer Connections, 1997--2025"* -- V-A load-independence, V-B
> national sums + device estimate, V-C connection-scaling test and coefficient
> history (these were IV-E/IV-F before the 7-Aug v5 renumbering). The paper
> ships two tiers with identical section numbering: a 14-pp journal version
> (OAJPE) and a 16-pp extended version (`main_extended_20260808.pdf`).
> Assertion battery: 47/47. Sign conventions: screen cells leading-positive;
> paper-facing cells and the saved figure Q-signed (Q < 0 = leading), flagged
> in place. See `../../replication/README.md` for the panel taxonomy and
> figure map.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

HERE = Path.cwd()
ROOT = HERE.resolve().parents[4]  # gridlytics repo root
SCREEN = ROOT / "clients/ea/harmonics/consultation-2026/analysis/resonance-screen"
rng = np.random.default_rng(20260806)
PASSED = 0

def ok(cond, msg):
    global PASSED
    assert cond, f"ASSERTION FAILED: {msg}"
    PASSED += 1
    print(f"  ok: {msg}")

icp = pd.read_csv(HERE / "data_raw" / "icp_by_rootnsp_20260806.csv.gz")
ok(list(icp.columns) == ["Month ended", "Region", "Participant code", "Entity name",
                         "ICP count", "ICP share (%)"], "raw columns as expected")
ok(icp["Month ended"].nunique() == 271, "271 months Dec 2003 - Jun 2026")
icp = icp[icp["Region"].astype(str).str.len() == 13].copy()   # drop 226 malformed rows
icp["poc"] = icp["Region"].str[:7]
icp["month"] = pd.to_datetime(icp["Month ended"])
per_poc = icp.groupby(["poc", "month"], as_index=False)["ICP count"].sum()
chk = per_poc[per_poc.month == "2025-06-30"]["ICP count"]
ok(int(chk.sum()) == 2339766, "national ICP total at 2025-06 = 2,339,766")
ok(len(chk) == 152, "152 distinct POCs at 2025-06 (159 region strings roll up across networks)")
per_poc["year"] = per_poc.month.dt.year
yr = (per_poc[per_poc.year.isin([2013, 2025])]
      .groupby(["poc", "year"])["ICP count"].mean().unstack())
yr.columns = ["N2013", "N2025"]
yr["dN"] = yr.N2025 - yr.N2013
yr["Nbar"] = (yr.N2013 + yr.N2025) / 2
yr.round(0).to_csv(HERE / "icp_per_poc_20260806.csv")
print(f"per-POC table: {len(yr)} POCs with 2013 and 2025 counts")

  ok: raw columns as expected
  ok: 271 months Dec 2003 - Jun 2026


  ok: national ICP total at 2025-06 = 2,339,766
  ok: 152 distinct POCs at 2025-06 (159 region strings roll up across networks)
per-POC table: 179 POCs with 2013 and 2025 counts


In [2]:
scr = pd.read_csv(SCREEN / "screen_by_gxp_year.csv")
fl = pd.read_csv(ROOT / "clients/ea/power-factor/replication/cache/contamination_flag.csv")
piv = scr.pivot_table(index="gxp_code", columns="year", values="qc_night_med")
bal = piv.dropna(subset=[2013, 2025]).copy()
bal["dQ"] = bal[2025] - bal[2013]
clean_codes = set(fl.loc[fl.contaminated == 0, "gxp"])
d_all = bal.join(yr, how="inner")
d = d_all[d_all.index.isin(clean_codes)].dropna(subset=["dQ", "Nbar", "dN"]).copy()
X, XdN, Y = d.Nbar.to_numpy(), d.dN.to_numpy(), d.dQ.to_numpy()
n = len(d)
ok(n == 83, "clean joined cohort n = 83")
ok(len(d_all) == 113, "all-flag joined cohort n = 113")

  ok: clean joined cohort n = 83
  ok: all-flag joined cohort n = 113


In [3]:
def ols(y, cols):
    A = np.column_stack([np.ones(len(y))] + cols)
    beta, *_ = np.linalg.lstsq(A, y, rcond=None)
    resid = y - A @ beta
    sse = float(resid @ resid)
    sst = float(((y - y.mean()) ** 2).sum())
    return beta, 1 - sse / sst, len(y) * np.log(sse / len(y)) + 2 * A.shape[1], resid

V = 1e6  # MVAr/ICP -> VAr/ICP
sse0 = float(((Y - Y.mean()) ** 2).sum())
aic0 = n * np.log(sse0 / n) + 2
b1, r2, aic1, resid1 = ols(Y, [X])
slope = b1[1] * V
boots = np.array([ols(Y[i], [X[i]])[0][1] for i in (rng.integers(0, n, (4000, n)))]) * V
ci = np.percentile(boots, [2.5, 97.5])
ii, jj = np.triu_indices(n, 1)
ts = float(np.median(((Y[jj] - Y[ii]) / (X[jj] - X[ii]))[np.isfinite((Y[jj]-Y[ii])/(X[jj]-X[ii]))])) * V
origin = float((X @ Y) / (X @ X)) * V
print(f"M1: dQ = {b1[0]:.2f} + {slope:.1f}e-6 * Nbar   R2={r2:.3f}")
print(f"    bootstrap 95% CI [{ci[0]:.1f}, {ci[1]:.1f}] VAr/ICP | Theil-Sen {ts:.1f} | through-origin {origin:.1f}")
print(f"    null (mean-only) AIC {aic0:.1f} vs M1 AIC {aic1:.1f}  (dAIC {aic0-aic1:.0f})")
ok(abs(slope - 252.8) < 1.0, "M1 slope 252.8 VAr/ICP (+/-1)")
ok(abs(b1[0]) < 0.2, "intercept indistinguishable from zero (<0.2 MVAr)")
ok(abs(r2 - 0.769) < 0.005, "R2 = 0.769 (+/-0.005)")
ok(ci[0] > 150 and abs(ci[0] - 191.8) < 5, "CI lower ~192, excludes 0 by a distance")
ok(abs(ci[1] - 291.4) < 5, "CI upper ~291")
ok(aic0 - aic1 > 100, "artefact null loses by dAIC > 100")
ok(abs(ts - 263.5) < 3, "Theil-Sen agrees (~264)")
ok(abs(origin - 251.0) < 3, "through-origin agrees (~251)")
ok(50 <= slope <= 400, "slope inside pre-registered band [50, 400]")

M1: dQ = -0.05 + 252.8e-6 * Nbar   R2=0.769
    bootstrap 95% CI [191.8, 291.4] VAr/ICP | Theil-Sen 263.5 | through-origin 251.0
    null (mean-only) AIC 222.1 vs M1 AIC 102.5  (dAIC 120)
  ok: M1 slope 252.8 VAr/ICP (+/-1)
  ok: intercept indistinguishable from zero (<0.2 MVAr)
  ok: R2 = 0.769 (+/-0.005)
  ok: CI lower ~192, excludes 0 by a distance
  ok: CI upper ~291
  ok: artefact null loses by dAIC > 100
  ok: Theil-Sen agrees (~264)
  ok: through-origin agrees (~251)
  ok: slope inside pre-registered band [50, 400]


In [4]:
b2, r2_2, aic2, _ = ols(Y, [X, XdN])
corr = float(np.corrcoef(X, XdN)[0, 1])
print(f"M2: a={b2[1]*V:.1f} VAr/ICP existing, b={b2[2]*V:.1f} VAr per NEW ICP, R2={r2_2:.3f}, corr(Nbar,dN)={corr:.2f}")
ok(abs(b2[2] * V) < 60, "new-connection term small (<60 VAr) - accumulation dominates")
ok(abs(corr - 0.72) < 0.03, "Nbar-dN collinearity 0.72 (interpret M2 cautiously)")
ok(aic2 > aic1, "M2 does not beat M1 on AIC")

loo = np.array([ols(np.delete(Y, i), [np.delete(X, i)])[0][1] for i in range(n)]) * V
print(f"leave-one-out slope range: [{loo.min():.1f}, {loo.max():.1f}]"
      f" (min = drop {d.index[np.argmin(loo)]}, max = drop {d.index[np.argmax(loo)]})")
ok(loo.min() > 230 and loo.max() < 280, "LOO slopes all within [230, 280] - no single bus drives the result")
ipen = list(d.index).index("PEN0331")
ok(abs(loo[ipen] - 239.1) < 1, "dropping high-leverage PEN0331 -> 239 (-5% only)")

M2: a=245.7 VAr/ICP existing, b=37.8 VAr per NEW ICP, R2=0.769, corr(Nbar,dN)=0.72
  ok: new-connection term small (<60 VAr) - accumulation dominates
  ok: Nbar-dN collinearity 0.72 (interpret M2 cautiously)
  ok: M2 does not beat M1 on AIC
leave-one-out slope range: [239.1, 269.6] (min = drop PEN0331, max = drop TAK0331)
  ok: LOO slopes all within [230, 280] - no single bus drives the result
  ok: dropping high-leverage PEN0331 -> 239 (-5% only)


In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

d["resid"] = resid1
fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.scatter(X / 1000, Y, s=22, c="0.15", alpha=0.75, lw=0, zorder=3)
xx = np.linspace(0, X.max() * 1.04, 50)
ax.plot(xx / 1000, b1[0] + b1[1] * xx, c="0.45", lw=1.4, zorder=2)
for code in d.reindex(d.resid.abs().sort_values(ascending=False).index).head(4).index.tolist() + ["PEN0331"]:
    r = d.loc[code]
    ax.annotate(code, (r.Nbar / 1000, r.dQ), textcoords="offset points",
                xytext=(5, 4), fontsize=7, color="0.35")
ax.set_xlabel("Mean ICP count behind GXP, 2013-25 (thousands)")
ax.set_ylabel("Rise in night-median net leading MVAr, 2013-25")
ax.set_title(f"Clean cohort (n={n}): slope {slope:.0f} VAr/ICP "
             f"[95% CI {ci[0]:.0f}-{ci[1]:.0f}], $R^2$={r2:.2f}, intercept~0", fontsize=10)
ax.grid(True, lw=0.3, alpha=0.5)
fig.tight_layout()
fig.savefig(HERE / "icp_regression_scatter_20260806.png", dpi=150)
fig.savefig(HERE / "icp_regression_scatter_20260806.pdf")
print("figure saved (working style - restyle before any paper use)")
plt.show()

figure saved (working style - restyle before any paper use)


## National sums, the bottom-up device estimate, and consistency (added 7 Aug 2026)

The IEEE paper (Section V-B; IV-E before the v5 renumbering) quotes the size of the organic rise and compares it with a bottom-up order-of-magnitude estimate of the device-fleet standing capacitance ("Fermi estimate" in the working docs). Until now those numbers lived only in the 6-Aug working documents (harmonics `PROXY_DECOMPOSITION_20260806.md`); the cells below make them notebook-asserted. The national sums need no ICP join, so they run on the full clean endpoint panel (n = 90, vs the 83 joined above). Sign convention in these cells stays leading-positive (screen convention); the paper flips to its own Q < 0 = leading. The last cell regenerates the scatter in the paper's style and sign convention.

In [6]:
# (a) Clean-panel national sums: every clean bus with the night-median defined in both
# 2013 and 2025 - no ICP join required, so n = 90 (the 83 above are these minus join losses).
cb = bal[bal.index.isin(clean_codes)]
s13, s25 = float(cb[2013].sum()), float(cb[2025].sum())
swing = s25 - s13
dq90 = cb[2025] - cb[2013]
risers, fallers = float(dq90[dq90 > 0].sum()), float(dq90[dq90 < 0].sum())
pivp = scr.pivot_table(index="gxp_code", columns="year", values="qc_p99")
cbp = pivp.dropna(subset=[2013, 2025])
cbp = cbp[cbp.index.isin(clean_codes)]
swing_p99 = float(cbp[2025].sum() - cbp[2013].sum())
HH_M = 2.0  # NZ households, millions (Stats NZ order; the device estimate's denominator)
per_hh = sorted([swing_p99 / HH_M, swing / HH_M, risers / HH_M])  # MVAr per M-hh = VAr/hh
print(f"clean endpoint panel n={len(cb)}: night-median {s13:+.1f} (2013) -> {s25:+.1f} MVAr (2025), swing {swing:+.1f}")
print(f"  risers {risers:+.1f} / fallers {fallers:+.1f}; p99-estimator swing {swing_p99:+.1f} (n={len(cbp)})")
print(f"  per household ({HH_M:.1f}M hh): {per_hh[0]:.0f}-{per_hh[-1]:.0f} VAr/hh across estimators")
ok(len(cb) == 90 and len(cbp) == 90, "clean endpoint panel n = 90 (both estimators)")
ok(abs(s13 + 65.7) < 1 and abs(s25 - 226.2) < 1 and abs(swing - 291.8) < 1,
   "night-median sums: -66 MVAr (2013) -> +226 MVAr (2025), swing +292")
ok(abs(risers - 322.7) < 1 and abs(fallers + 30.9) < 1 and abs(swing_p99 - 226.9) < 1,
   "risers +323 / fallers -31; p99-estimator swing +227")
ok(112 < per_hh[0] < 115 and 159 < per_hh[-1] < 163,
   "per-household equivalents span 113-161 VAr/hh across estimators")

clean endpoint panel n=90: night-median -65.7 (2013) -> +226.2 MVAr (2025), swing +291.8
  risers +322.7 / fallers -30.9; p99-estimator swing +226.9 (n=90)
  per household (2.0M hh): 113-161 VAr/hh across estimators
  ok: clean endpoint panel n = 90 (both estimators)
  ok: night-median sums: -66 MVAr (2013) -> +226 MVAr (2025), swing +292
  ok: risers +323 / fallers -31; p99-estimator swing +227
  ok: per-household equivalents span 113-161 VAr/hh across estimators


In [7]:
# (a2) Full clean endpoint cohort, archive-native (D1, Dave 7 Aug eve). The screen table
# above is fault-level-constrained (90 sites); the pf half-hourly archive itself defines
# the full clean cohort with both endpoints metered in BOTH windows: n = 114 - the same
# cohort as the parallel-shift (unmasking) test, whose headline numbers must reproduce
# here as a cross-validation (they are the paper's V-A load-independence figures, until
# now asserted only in the harmonics workings notebook - native from this cell on).
ANA = ROOT / "clients/ea/power-factor/data/analysis"
NIGHT, PEAK = range(1, 13), range(35, 40)
arows = {}
for y in (2013, 2025):
    adf = pd.read_parquet(ANA / f"{y}_pf_by_gxp.parquet",
                          columns=["trading_period", "gxp_code", "P", "Q"])
    adf["lead"] = -adf.Q
    arows[y] = pd.concat([
        adf[adf.trading_period.isin(NIGHT)].groupby("gxp_code")["lead"].median().rename("night"),
        adf[adf.trading_period.isin(PEAK)].groupby("gxp_code")["lead"].median().rename("peak"),
        adf[adf.trading_period.isin(PEAK)].groupby("gxp_code")["P"].median().rename("peak_P"),
        adf.groupby("gxp_code")["lead"].quantile(0.99).rename("p99")], axis=1)
a13, a25 = arows[2013], arows[2025]
cohort114 = [g for g in a13.dropna().index.intersection(a25.dropna().index) if g in clean_codes]
n114 = len(cohort114)
s13_114 = float(a13.loc[cohort114, "night"].sum())
s25_114 = float(a25.loc[cohort114, "night"].sum())
swing114 = s25_114 - s13_114
dq114 = a25.loc[cohort114, "night"] - a13.loc[cohort114, "night"]
risers114, fallers114 = float(dq114[dq114 > 0].sum()), float(dq114[dq114 < 0].sum())
swing114_p99 = float(a25.loc[cohort114, "p99"].sum() - a13.loc[cohort114, "p99"].sum())
per_hh114 = sorted([swing114_p99 / HH_M, swing114 / HH_M, risers114 / HH_M])
dk114 = a25.loc[cohort114, "peak"] - a13.loc[cohort114, "peak"]
ratio114 = float((dk114 / dq114.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).median())
pload114 = float((a25.loc[cohort114, "peak_P"] / a13.loc[cohort114, "peak_P"] - 1).median()) * 100
print(f"archive-native clean cohort n={n114}: night-median {s13_114:+.1f} (2013) -> {s25_114:+.1f} MVAr (2025), deepening {swing114:+.1f}")
print(f"  risers {risers114:+.1f} / fallers {fallers114:+.1f}; p99-estimator deepening {swing114_p99:+.1f}")
print(f"  per household ({HH_M:.1f}M hh): {per_hh114[0]:.0f}-{per_hh114[-1]:.0f} VAr/hh across estimators")
print(f"  parallel-shift cross-check: night rise med {dq114.median():+.2f}, peak rise med {dk114.median():+.2f}, "
      f"ratio med {ratio114:.2f}, peak load change {pload114:+.1f}%")
ok(n114 == 114, "archive-native clean endpoint cohort n = 114 (night + peak medians, both years)")
ok(abs(dq114.median() - 2.25) < 0.03 and abs(dk114.median() - 2.42) < 0.03 and abs(ratio114 - 1.06) < 0.03
   and abs(pload114 - 3.4) < 0.3,
   "parallel-shift test reproduces natively: +2.25 night / +2.42 peak / ratio 1.06 / peak load +3.4%")
ok(abs(s13_114 + 187.5) < 1 and abs(s25_114 - 181.5) < 1 and abs(swing114 - 369.0) < 1,
   "114-site night-median sums: -188 (2013) -> +182 MVAr (2025), deepening +369")
ok(abs(risers114 - 427.4) < 1 and abs(fallers114 + 58.4) < 1 and abs(swing114_p99 - 313.7) < 1.5,
   "risers +427 / fallers -58; p99-estimator deepening +314")
ok(155 < per_hh114[0] < 159 and 212 < per_hh114[-1] < 216,
   "per-household 157-214 VAr/hh across estimators (2.0M hh)")
ok(150 < swing114_p99 and swing114 < 600,
   "114-site deepening (+314 to +369) sits inside the 150-600 MVAr estimate band; per-household "
   "sits at the upper edge of the 100-200 central band")

archive-native clean cohort n=114: night-median -187.5 (2013) -> +181.5 MVAr (2025), deepening +369.0
  risers +427.4 / fallers -58.4; p99-estimator deepening +313.7
  per household (2.0M hh): 157-214 VAr/hh across estimators
  parallel-shift cross-check: night rise med +2.25, peak rise med +2.42, ratio med 1.06, peak load change +3.4%
  ok: archive-native clean endpoint cohort n = 114 (night + peak medians, both years)
  ok: parallel-shift test reproduces natively: +2.25 night / +2.42 peak / ratio 1.06 / peak load +3.4%
  ok: 114-site night-median sums: -188 (2013) -> +182 MVAr (2025), deepening +369
  ok: risers +427 / fallers -58; p99-estimator deepening +314
  ok: per-household 157-214 VAr/hh across estimators (2.0M hh)
  ok: 114-site deepening (+314 to +369) sits inside the 150-600 MVAr estimate band; per-household sits at the upper edge of the 100-200 central band


In [8]:
# (a3) The coefficient's history (Dave, 7 Aug night: "can we look at how this has changed
# in time"). Year-by-year cross-sectional slope beta(t) of night-median leading LEVEL on
# ICP count, same cohort every year (the regression's 83 minus 2 without full 2009-2025
# presence). Levels, so each year's slope is the whole connection-scaling class; its
# MOVEMENT is the accumulation. The archive product starts 2009 - four pre-window years,
# enough to establish the zero baseline.
Nyr = per_poc.copy()
Nyr["year"] = Nyr.month.dt.year
Nyr = Nyr.groupby(["poc", "year"])["ICP count"].mean().unstack()
lev = {}
for y in range(2009, 2026):
    adf = pd.read_parquet(ANA / f"{y}_pf_by_gxp.parquet", columns=["trading_period", "gxp_code", "Q"])
    s = adf[adf.gxp_code.isin(d.index) & adf.trading_period.isin(NIGHT)]
    lev[y] = (-s.Q).groupby(s.gxp_code).median()
lev = pd.DataFrame(lev)
present81 = lev.dropna().index
beta_rows = []
for y in range(2009, 2026):
    Ny = Nyr.loc[[p for p in present81 if p in Nyr.index], y].dropna()
    gg = [g for g in present81 if g in Ny.index]
    Xb, Yb = Ny.loc[gg].to_numpy(), lev.loc[gg, y].to_numpy()
    Ab = np.column_stack([np.ones(len(Xb)), Xb])
    bb, *_ = np.linalg.lstsq(Ab, Yb, rcond=None)
    bboots = [np.linalg.lstsq(Ab[i], Yb[i], rcond=None)[0][1] * 1e6
              for i in rng.integers(0, len(Xb), (1000, len(Xb)))]
    lo_b, hi_b = np.percentile(bboots, [2.5, 97.5])
    beta_rows.append((y, len(gg), bb[1] * 1e6, lo_b, hi_b))
bt = pd.DataFrame(beta_rows, columns=["year", "n", "beta", "lo", "hi"]).set_index("year")
print(bt.round(1).to_string())
d_beta = bt.beta.loc[2025] - bt.beta.loc[2013]
r1 = (bt.beta.loc[2013] - bt.beta.loc[2009]) / 4
r2_h = (bt.beta.loc[2019] - bt.beta.loc[2013]) / 6
r3 = (bt.beta.loc[2025] - bt.beta.loc[2019]) / 6
print(f"\nbeta(2025)-beta(2013) = {d_beta:.0f} VAr/ICP vs two-endpoint slope {slope:.0f}")
print(f"accumulation rate: 2009-13 {r1:.1f} | 2013-19 {r2_h:.1f} | 2019-25 {r3:.1f} VAr/ICP/yr")
ok(len(present81) == 81 and int(bt.n.min()) == 81, "beta(t) cohort: 81 of the 83 present in every year 2009-2025")
ok(bt.lo.loc[2009] < 0 < bt.hi.loc[2009] and abs(bt.beta.loc[2009]) < 50,
   "2009 baseline: coefficient indistinguishable from zero")
ok(bool((bt.beta.diff().loc[2012:] > 0).all()),
   "the coefficient rises every single year from 2012 onward - smooth, no steps")
ok(bool((bt.lo.loc[2018:] > 0).all()), "interval excludes zero from 2018 at the latest")
ok(abs(bt.beta.loc[2013] - 27.3) < 2 and abs(bt.beta.loc[2019] - 131.1) < 2 and abs(bt.beta.loc[2025] - 273.8) < 2,
   "beta path pinned: 27 (2013) -> 131 (2019) -> 274 (2025) VAr/ICP")
ok(abs(d_beta - 246) < 3 and abs(d_beta - slope) < 15,
   "independent cross-check: beta(2025)-beta(2013) = 246 vs the two-endpoint slope 253 (within 3%)")
ok(abs(r2_h - 17.3) < 1 and abs(r3 - 23.8) < 1 and r3 > r2_h > r1,
   "accumulation rate accelerates: ~13 -> ~17 -> ~24 VAr/ICP/yr across the three spans")

       n   beta     lo     hi
year                         
2009  81  -24.6 -114.9   40.3
2010  81  -30.6  -93.4   41.6
2011  81  -15.1  -76.3   70.4
2012  81   13.9  -57.5  118.2
2013  81   27.3  -35.8  137.7
2014  81   33.4  -33.0  145.5
2015  81   46.4  -23.5  155.6
2016  81   67.6  -10.9  171.8
2017  81   95.5   25.8  195.4
2018  81  114.0   42.1  216.8
2019  81  131.1   62.9  227.6
2020  81  177.2  109.0  277.9
2021  81  209.9  142.3  310.1
2022  81  223.2  155.6  309.7
2023  81  230.5  177.5  300.5
2024  81  250.5  192.8  330.8
2025  81  273.8  213.8  356.0

beta(2025)-beta(2013) = 246 VAr/ICP vs two-endpoint slope 253
accumulation rate: 2009-13 13.0 | 2013-19 17.3 | 2019-25 23.8 VAr/ICP/yr
  ok: beta(t) cohort: 81 of the 83 present in every year 2009-2025
  ok: 2009 baseline: coefficient indistinguishable from zero
  ok: the coefficient rises every single year from 2012 onward - smooth, no steps
  ok: interval excludes zero from 2018 at the latest
  ok: beta path pinned: 27 (201

In [9]:
# (b) The bottom-up order-of-magnitude device estimate - the hypothesis's testable
# prediction. Inputs are physics and datasheet-grade values, not NZ bench measurements:
# hypothesis-grade [H]. X-capacitor standing VAr at 230 V, 50 Hz: Q = omega * C * V^2.
OMEGA, V230 = 2 * np.pi * 50, 230.0
xcap = {uF: OMEGA * uF * 1e-6 * V230**2 for uF in (0.1, 0.47, 1.0)}
print("X-cap VAr at 230 V, 50 Hz:", {k: round(v, 1) for k, v in xcap.items()})
ok(abs(xcap[0.1] - 1.66) < 0.05 and abs(xcap[0.47] - 7.81) < 0.1 and abs(xcap[1.0] - 16.62) < 0.1,
   "X-capacitor standing VAr: 0.1 uF = 1.7, 0.47 uF = 7.8, 1.0 uF = 16.6")
res_lo, res_hi = HH_M * 20 * 2, HH_M * 40 * 4  # 20-40 always-connected devices x 2-4 VAr -> MVAr
com_lo, com_hi = 50.0, 300.0                   # commercial/industrial leg (same order; weakest data)
nat_lo, nat_hi = res_lo + com_lo, res_hi + com_hi
BAND = (150.0, 600.0)     # quoted national band (envelope tightened by judgment; central ~300)
PRED_HH = (100.0, 200.0)  # central per-household prediction band (<-> 200-400 MVAr national)
print(f"residential {res_lo:.0f}-{res_hi:.0f} MVAr + commercial {com_lo:.0f}-{com_hi:.0f}"
      f" = envelope {nat_lo:.0f}-{nat_hi:.0f}; quoted band {BAND[0]:.0f}-{BAND[1]:.0f}, central ~300")
ok((res_lo, res_hi) == (80.0, 320.0) and nat_lo <= BAND[0] and BAND[1] <= nat_hi,
   "residential 80-320 MVAr; national envelope (130-620) brackets the quoted 150-600 band")

X-cap VAr at 230 V, 50 Hz: {0.1: 1.7, 0.47: 7.8, 1.0: 16.6}
  ok: X-capacitor standing VAr: 0.1 uF = 1.7, 0.47 uF = 7.8, 1.0 uF = 16.6
residential 80-320 MVAr + commercial 50-300 = envelope 130-620; quoted band 150-600, central ~300
  ok: residential 80-320 MVAr; national envelope (130-620) brackets the quoted 150-600 band


In [10]:
# (b2) Tier-1 heat-pump slice of the residential estimate (added 7 Aug 2026, Dave's ask).
# Stock side [V/I]: heat pumps used in 66.8% of dwellings at the 2023 Census (47.3% in 2018)
#   - verified via EHINZ (Massey) indicator page reporting Stats NZ Census data, 7 Aug 2026;
#   units per using dwelling 1.2-1.6 [I assumption]; cross-check: EECA/Figure.NZ sales data
#   ~150k units/yr average since 2013 (record ~240k in 2023) -> ~1.9M sold in-window alone.
# Electrical side [H]: standing across-line filter capacitance 0.3-2.2 uF per unit
#   (datasheet-grade appliance EMI-filter values, NOT NZ-bench-measured - the bench
#   measurement stays the parked check).
PEN23, PEN18 = 0.668, 0.473
u_lo, u_hi = 1.2, 1.6
n_lo, n_hi = HH_M * PEN23 * u_lo, HH_M * PEN23 * u_hi        # millions of residential units
q_lo, q_hi = OMEGA * 0.3e-6 * V230**2, OMEGA * 2.2e-6 * V230**2   # VAr per unit
hp_lo, hp_hi = n_lo * q_lo, n_hi * q_hi                       # MVAr national residential
hp_central = HH_M * PEN23 * 1.4 * (OMEGA * 0.9e-6 * V230**2)
print(f"residential heat-pump units: {n_lo:.2f}-{n_hi:.2f}M "
      f"(census 66.8% x {u_lo}-{u_hi} units/dwelling; sales integral ~{0.150*13:.1f}M since 2013)")
print(f"per-unit standing: {q_lo:.1f}-{q_hi:.1f} VAr (0.3-2.2 uF at 230 V)")
print(f"heat-pump slice: {hp_lo:.0f}-{hp_hi:.0f} MVAr national residential (central ~{hp_central:.0f});"
      f" per household {hp_lo/HH_M:.0f}-{hp_hi/HH_M:.0f} VAr/hh")
ok(abs(q_lo - 5.0) < 0.1 and abs(q_hi - 36.6) < 0.3, "per-unit standing VAr band 5.0-36.6 (0.3-2.2 uF)")
ok(1.55 < n_lo < 1.65 and 2.1 < n_hi < 2.2 and abs(0.150 * 13 - 1.95) < 0.01,
   "units 1.60-2.14M; EECA-sales cross-check ~1.95M sold 2013-25 (same order)")
ok(7.9 <= hp_lo <= 9 and 77 <= hp_hi <= 80, "heat-pump slice ~8-78 MVAr, central ~28")
ok(hp_hi <= res_lo, "even the slice's UPPER bound sits at/below the residential envelope's FLOOR "
                    "(80 MVAr) - heat pumps are a material but minority class")
ok(hp_hi / HH_M < 113, "at most ~39 VAr/hh - the heat-pump fleet alone cannot carry the measured "
                       "113-161 VAr/hh rise; the swarm of smaller supplies does")

residential heat-pump units: 1.60-2.14M (census 66.8% x 1.2-1.6 units/dwelling; sales integral ~1.9M since 2013)
per-unit standing: 5.0-36.6 VAr (0.3-2.2 uF at 230 V)
heat-pump slice: 8-78 MVAr national residential (central ~28); per household 4-39 VAr/hh
  ok: per-unit standing VAr band 5.0-36.6 (0.3-2.2 uF)
  ok: units 1.60-2.14M; EECA-sales cross-check ~1.95M sold 2013-25 (same order)
  ok: heat-pump slice ~8-78 MVAr, central ~28
  ok: even the slice's UPPER bound sits at/below the residential envelope's FLOOR (80 MVAr) - heat pumps are a material but minority class
  ok: at most ~39 VAr/hh - the heat-pump fleet alone cannot carry the measured 113-161 VAr/hh rise; the swarm of smaller supplies does


In [11]:
# (c) The consistency comparison the paper states, plus the residual-sign check behind
# its residual sentence (industrial-mix buses under-rise relative to their connection count).
ok(nat_lo < swing_p99 and swing < nat_hi and BAND[0] < swing_p99 and swing < BAND[1],
   "measured national rise (+227 to +292 MVAr) is bracketed by the device estimate")
ok(PRED_HH[0] <= per_hh[0] and per_hh[-1] <= PRED_HH[1],
   "measured 113-161 VAr/hh sits inside the predicted 100-200 VAr/hh central band")
ok(float(d.loc["TAK0331", "resid"]) < 0 and float(d.loc["WIR0331", "resid"]) < 0,
   "industrial-mix Takanini and Wiri sit below the fitted line (under-rise)")
print(f"consistency: measured rise {swing_p99:+.0f}..{swing:+.0f} MVAr "
      f"({per_hh[0]:.0f}-{per_hh[-1]:.0f} VAr/hh)  vs  bottom-up {BAND[0]:.0f}-{BAND[1]:.0f} MVAr "
      f"({PRED_HH[0]:.0f}-{PRED_HH[1]:.0f} VAr/hh central)")

  ok: measured national rise (+227 to +292 MVAr) is bracketed by the device estimate
  ok: measured 113-161 VAr/hh sits inside the predicted 100-200 VAr/hh central band
  ok: industrial-mix Takanini and Wiri sit below the fitted line (under-rise)
consistency: measured rise +227..+292 MVAr (113-161 VAr/hh)  vs  bottom-up 150-600 MVAr (100-200 VAr/hh central)


In [12]:
# Paper figure, TWO PANELS (7 Aug night): A = the two-endpoint scatter; B = the
# coefficient's history beta(t). Paper sign convention (Q < 0 = leading) throughout, so
# changes and coefficients plot negative. House style; saved into ../replication/figures/
# where main.tex pulls its figures from. Working scatter above unchanged.
FIGDIR = ROOT / "clients/ea/power-factor/replication/figures"
EA_BLUE, GREEN = "#003366", "#2ca02c"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 130, "font.size": 11,
    "axes.titlesize": 12, "axes.titlecolor": EA_BLUE, "axes.grid": True,
    "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "figure.facecolor": "white",
})
fig, (axA, axB) = plt.subplots(2, 1, figsize=(7.2, 9.0))
axA.axhline(0, color="0.6", lw=0.8, ls="--", zorder=1)
axA.scatter(X / 1000, -Y, s=26, c=EA_BLUE, alpha=0.75, lw=0, zorder=3)
xx = np.linspace(0, X.max() * 1.04, 50)
axA.plot(xx / 1000, -(b1[0] + b1[1] * xx), c=GREEN, lw=1.8, zorder=2)
axA.set_xlabel("Connections served (mean of 2013 and 2025 ICP count, thousands)")
axA.set_ylabel("Change in median overnight reactive power\n2013-2025 (MVAr)")
axA.set_title("A. The overnight deepening scales with connections served")
axA.annotate(f"slope $-${abs(slope):.0f} VAr per connection\n"
             f"bootstrap 95% CI $-${abs(ci[1]):.0f} to $-${abs(ci[0]):.0f}\n"
             f"intercept $\\approx$ 0,  $R^2$ = {r2:.2f}",
             xy=(0.97, 0.95), xycoords="axes fraction", ha="right", va="top", fontsize=10)
axB.axhline(0, color="0.6", lw=0.8, ls="--", zorder=1)
axB.fill_between(bt.index, -bt.hi, -bt.lo, color=EA_BLUE, alpha=0.15, lw=0, zorder=2)
axB.plot(bt.index, -bt.beta, "o-", c=EA_BLUE, lw=1.8, ms=4, zorder=3)
axB.set_xlabel("Year")
axB.set_ylabel("Standing VAr per connection\n(cross-sectional coefficient)")
axB.set_title("B. The coefficient's history: near zero until 2012,\ndeepening every year since")
axB.annotate("accumulation rate\n2009-13 $\\approx$ 13, 2013-19 $\\approx$ 17,\n"
             "2019-25 $\\approx$ 24 VAr per connection per year",
             xy=(0.03, 0.06), xycoords="axes fraction", va="bottom", fontsize=10)
fig.tight_layout()
fig.savefig(FIGDIR / "06_connection_scaling.png")
print(f"paper figure (2 panels) -> {FIGDIR / '06_connection_scaling.png'}")
plt.show()

paper figure (2 panels) -> /home/dave/gridlytics/clients/ea/power-factor/replication/figures/06_connection_scaling.png


In [13]:
res = {
 "n_clean_joined": n, "n_all_joined": int(len(d_all)),
 "M0_null": {"aic": round(aic0, 1)},
 "M1": {"slope_var_per_icp": round(slope, 1),
        "ci95_var_per_icp": [round(float(c), 1) for c in ci],
        "intercept_mvar": round(float(b1[0]), 2), "r2": round(r2, 3), "aic": round(aic1, 1)},
 "M1_theilsen_var_per_icp": round(ts, 1),
 "M1_through_origin_var_per_icp": round(origin, 1),
 "M2": {"a_var_per_icp": round(b2[1] * V, 1), "b_var_per_new_icp": round(b2[2] * V, 1),
        "r2": round(r2_2, 3), "aic": round(aic2, 1), "corr_Nbar_dN": round(corr, 2)},
 "loo_slope_range_var_per_icp": [round(float(loo.min()), 1), round(float(loo.max()), 1)],
 "prediction_band_var_per_icp": [50, 400],
 "verdict": "SUPPORTED (pre-specified rule): slope in band, CI excludes 0, R2 >> 0.15",
 "beta_t_history": {
     "n": 81, "years": [2009, 2025],
     "beta_var_per_icp": {str(y): round(float(bt.beta.loc[y]), 1) for y in bt.index},
     "endpoint_diff_var_per_icp": round(float(d_beta), 1),
     "accum_rate_var_per_icp_yr": {"2009-13": round(float(r1), 1), "2013-19": round(float(r2_h), 1),
                                   "2019-25": round(float(r3), 1)},
     "notes": "cross-sectional level slope per year, same 81 buses; CI excludes 0 from 2018 "
              "at the latest; monotone rise from 2012; level slope = whole connection-scaling class"},
 "national_sums_clean90": {
     "n": int(len(cb)), "night_med_2013_mvar": round(s13, 1), "night_med_2025_mvar": round(s25, 1),
     "swing_mvar": round(swing, 1), "risers_mvar": round(risers, 1), "fallers_mvar": round(fallers, 1),
     "swing_p99_mvar": round(swing_p99, 1),
     "per_household_var": [round(per_hh[0], 1), round(per_hh[-1], 1)], "households_m": HH_M,
     "note": "fault-level-constrained screen-table subset; regression panel derives from this; "
             "the PAPER quotes the 114-site archive-native sums below (D1, 7 Aug)"},
 "national_sums_clean114": {
     "n": n114, "night_med_2013_mvar": round(s13_114, 1), "night_med_2025_mvar": round(s25_114, 1),
     "swing_mvar": round(swing114, 1), "risers_mvar": round(risers114, 1),
     "fallers_mvar": round(fallers114, 1), "swing_p99_mvar": round(swing114_p99, 1),
     "per_household_var": [round(per_hh114[0], 1), round(per_hh114[-1], 1)], "households_m": HH_M,
     "parallel_shift_native": {"night_rise_med_mvar": round(float(dq114.median()), 2),
                               "peak_rise_med_mvar": round(float(dk114.median()), 2),
                               "ratio_med": round(ratio114, 2),
                               "peak_load_chg_pct": round(pload114, 1)}},
 "device_estimate": {
     "xcap_var_at_230v": {str(k): round(v, 2) for k, v in xcap.items()},
     "residential_mvar": [res_lo, res_hi], "commercial_mvar": [com_lo, com_hi],
     "national_envelope_mvar": [nat_lo, nat_hi], "quoted_band_mvar": list(BAND),
     "per_household_central_var": list(PRED_HH)},
 "heatpump_slice_tier1": {
     "census_penetration": {"2018": PEN18, "2023": PEN23},
     "residential_units_m": [round(n_lo, 2), round(n_hi, 2)],
     "per_unit_var": [round(q_lo, 1), round(q_hi, 1)], "per_unit_uF": [0.3, 2.2],
     "slice_mvar": [round(hp_lo, 1), round(hp_hi, 1)], "central_mvar": round(hp_central, 0),
     "per_household_var_max": round(hp_hi / HH_M, 1),
     "verdict": "material but minority: upper bound ~ residential envelope floor; cannot carry the "
                "measured per-household rise (113-161 on the 90-panel; 157-214 on the 114-panel) - "
                "the small-supply swarm does"},
 "consistency": "measured 2013-25 deepening (114-site panel: +314 to +369 MVAr; 157-214 VAr/hh) "
                "inside the bottom-up 150-600 MVAr band, per-household at the upper edge of the "
                "100-200 central band",
 "paper_figure": "replication/figures/06_connection_scaling.png (2 panels: scatter + beta_t)",
}
(HERE / "icp_regression_results_20260806.json").write_text(json.dumps(res, indent=1))
print(json.dumps(res, indent=1))
print(f"\nASSERTIONS PASSED: {PASSED}/{PASSED}")

{
 "n_clean_joined": 83,
 "n_all_joined": 113,
 "M0_null": {
  "aic": 222.1
 },
 "M1": {
  "slope_var_per_icp": 252.8,
  "ci95_var_per_icp": [
   191.8,
   291.4
  ],
  "intercept_mvar": -0.05,
  "r2": 0.769,
  "aic": 102.5
 },
 "M1_theilsen_var_per_icp": 263.5,
 "M1_through_origin_var_per_icp": 251.0,
 "M2": {
  "a_var_per_icp": 245.7,
  "b_var_per_new_icp": 37.8,
  "r2": 0.769,
  "aic": 104.3,
  "corr_Nbar_dN": 0.72
 },
 "loo_slope_range_var_per_icp": [
  239.1,
  269.6
 ],
 "prediction_band_var_per_icp": [
  50,
  400
 ],
 "verdict": "SUPPORTED (pre-specified rule): slope in band, CI excludes 0, R2 >> 0.15",
 "beta_t_history": {
  "n": 81,
  "years": [
   2009,
   2025
  ],
  "beta_var_per_icp": {
   "2009": -24.6,
   "2010": -30.6,
   "2011": -15.1,
   "2012": 13.9,
   "2013": 27.3,
   "2014": 33.4,
   "2015": 46.4,
   "2016": 67.6,
   "2017": 95.5,
   "2018": 114.0,
   "2019": 131.1,
   "2020": 177.2,
   "2021": 209.9,
   "2022": 223.2,
   "2023": 230.5,
   "2024": 250.5,
   "2025

## Verdict and caveats

**SUPPORTED under the pre-specified rule.** The twelve-year rise scales linearly with connections at ~253 VAr per ICP (95% CI 192-291), intercept indistinguishable from zero, R^2 = 0.77; robust estimators agree; leave-one-out spans only 239-270; the bus-level-artefact null loses by dAIC ~ 120.

**Caveats, in force.** (1) Across buses, ICP count is collinear with bus scale - strictly this demonstrates clean linear scaling with connections served; no plausible metering mechanism produces that with zero intercept, so the metering-registration rival is heavily disfavoured, not formally dead. (2) The slope is the whole connection-scaling class: device filter capacitance PLUS each connection's share of MV cable (LV cable is negligible at 400 V - Q scales with V^2). The split within the slope belongs to the LV distribution-transformer night measurement. (3) Sitting above the 100-200 VAr/household central device estimate is expected: commercial ICPs carry larger filter fleets and the cable share rides along. (4) Industrial-mix buses (Takanini, Wiri) under-rise; Frankton/Hamilton over-rise - sensible scatter, no pathology.

**Next:** LV distribution-transformer night measurement (isolates the device fleet: LV-side metering sees no transformer magnetising and no meaningful cable charging); EDB LV-monitor-fleet data request is the force-multiplied version. Paper use: FSR 2026 ([V]-grade scaling sentence), canary paper (mechanism correction), harmonics W6.

**Added 7 Aug 2026 (IEEE-paper drafting session):** the clean-panel national sums (n = 90: night-median -66 -> +226 MVAr, swing +292, risers +323 / fallers -31; p99 variant +227; 113-161 VAr per household at 2.0M households), the bottom-up device estimate as explicit arithmetic (X-cap 1.7-16.6 VAr; residential 80-320 MVAr; national ~150-600 central ~300; 100-200 VAr/hh central band), the consistency comparison, and the paper-styled figure (`../../replication/figures/06_connection_scaling.png`, paper sign convention). These were previously quotable only from `PROXY_DECOMPOSITION_20260806.md`; they are now notebook-asserted. Paper landing: Sections V-B / V-C of `../main.tex` (IV-E/IV-F before the v5 renumbering); registered in `../../replication/README.md`.

**Added 7 Aug 2026 pm - tier-1 heat-pump slice (Dave's ask).** Census penetration 47.3% (2018) -> 66.8% (2023) [V via EHINZ/Stats NZ; the jump sits inside the study window, mostly at existing homes - consistent with the accumulation-dominates finding]; 1.6-2.1M residential units (1.2-1.6 per using dwelling [I]; EECA-sales integral ~1.9M sold 2013-25 agrees); 0.3-2.2 uF standing filter capacitance per unit [H, datasheet-grade] -> 5-37 VAr/unit -> **slice ~8-78 MVAr national residential (central ~28), at most ~39 VAr/household**. Verdict: heat pumps are a material but MINORITY class - even the slice's upper bound sits at the residential envelope's floor and cannot carry the measured per-household rise; the swarm of 20-40 small always-connected supplies per household does. Heat pumps' dominance is in the ENERGY transition (see christchurch-case: highest-main-centre consumption), not the standing-capacitance mechanism. One sentence in paper V-B.

**Added 7 Aug 2026 eve - D1: archive-native 114-site sums (Dave's call after the review pass).** Cell (a2) computes the FULL clean endpoint cohort directly from the pf half-hourly archive (night TP 1-12 + peak TP 35-39 medians + full-year p99, 2013 and 2025): **n = 114; night-median -188 -> +182 MVAr, deepening +369 (risers +427 / fallers -58); p99 variant +314; 157-214 VAr per household** - these are now the numbers the PAPER quotes in V-B (the 90-site screen-table sums remain above: they define the regression panel and back the harmonics documents). Bonus: **the parallel-shift (unmasking) test now reproduces natively in this notebook** (+2.25 night / +2.42 peak / ratio 1.06 / peak load +3.4% on the same 114) - previously only the harmonics workings notebook carried it, which was the replication package's public-release blocker; that flag is resolved in `../../replication/README.md`. Battery **34 -> 40/40**.